Configuración general

In [0]:
# Databricks notebook source

CATALOG = "fintech_finpay"

SCHEMA_DEFAULT = "default"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"
SCHEMA_OBSERVABILITY = "observability"

VOLUME_NAME = "vol_landing"
LANDING_PATH = f"/Volumes/{CATALOG}/{SCHEMA_DEFAULT}/{VOLUME_NAME}"

GROUP_INGENIERIA = "ingenieria"
GROUP_RIESGO = "riesgo"
GROUP_AUDITORIA = "auditoria"

print("Catalog:", CATALOG)
print("Landing path:", LANDING_PATH)

CREATE CATALOG IF NOT EXISTS fintech_finpay;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS fintech_finpay;

Crear schemas

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS fintech_finpay.default;
CREATE SCHEMA IF NOT EXISTS fintech_finpay.bronze;
CREATE SCHEMA IF NOT EXISTS fintech_finpay.silver;
CREATE SCHEMA IF NOT EXISTS fintech_finpay.gold;
CREATE SCHEMA IF NOT EXISTS fintech_finpay.observability;

Crear volume de landing

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS fintech_finpay.default.vol_landing;

In [0]:

dbutils.fs.mkdirs(f"{LANDING_PATH}/transactions")
dbutils.fs.mkdirs(f"{LANDING_PATH}/merchants")
dbutils.fs.mkdirs(f"{LANDING_PATH}/users")
dbutils.fs.mkdirs(f"{LANDING_PATH}/metadata")
dbutils.fs.mkdirs(f"{LANDING_PATH}/schemas")
dbutils.fs.mkdirs(f"{LANDING_PATH}/checkpoints")
dbutils.fs.mkdirs(f"{LANDING_PATH}/bad_records")

display(dbutils.fs.ls(LANDING_PATH))

In [0]:
import json

path = f"{LANDING_PATH}/metadata/ingestion_archetypes.json"

with open(path, "r", encoding="utf-8") as file:
    data = json.load(file)

print(type(data))
print(len(data))
display(spark.createDataFrame(data))

Permisos sobre catálogo

In [0]:
%sql
GRANT USE CATALOG ON CATALOG fintech_finpay TO `ingenieria`;
GRANT USE CATALOG ON CATALOG fintech_finpay TO `riesgo`;
GRANT USE CATALOG ON CATALOG fintech_finpay TO `auditoria`;

Permisos para ingeniería


In [0]:
%sql
GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT
ON SCHEMA fintech_finpay.default
TO `ingenieria`;

GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT
ON SCHEMA fintech_finpay.bronze
TO `ingenieria`;

GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT
ON SCHEMA fintech_finpay.silver
TO `ingenieria`;

GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT
ON SCHEMA fintech_finpay.gold
TO `ingenieria`;

GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT
ON SCHEMA fintech_finpay.observability
TO `ingenieria`;

GRANT READ VOLUME, WRITE VOLUME
ON VOLUME fintech_finpay.default.vol_landing
TO `ingenieria`;

Permisos para riesgo

In [0]:
%sql
GRANT USE SCHEMA ON SCHEMA fintech_finpay.silver TO `riesgo`;
GRANT USE SCHEMA ON SCHEMA fintech_finpay.gold TO `riesgo`;

GRANT SELECT ON SCHEMA fintech_finpay.silver TO `riesgo`;
GRANT SELECT ON SCHEMA fintech_finpay.gold TO `riesgo`;

GRANT READ VOLUME
ON VOLUME fintech_finpay.default.vol_landing
TO `riesgo`;

Permisos para auditoría

In [0]:
%sql
GRANT USE SCHEMA ON SCHEMA fintech_finpay.gold TO `auditoria`;
GRANT USE SCHEMA ON SCHEMA fintech_finpay.observability TO `auditoria`;

GRANT SELECT ON SCHEMA fintech_finpay.gold TO `auditoria`;
GRANT SELECT ON SCHEMA fintech_finpay.observability TO `auditoria`;

Crear función de masking para PII

In [0]:
%sql
CREATE OR REPLACE FUNCTION fintech_finpay.default.mask_pii(value STRING)
RETURN
  CASE
    WHEN is_account_group_member('ingenieria') THEN value
    ELSE '***MASKED***'
  END;

Crear función de row-level security

In [0]:
%sql
CREATE OR REPLACE FUNCTION fintech_finpay.default.users_row_filter(country STRING)
RETURN
  CASE
    WHEN is_account_group_member('ingenieria') THEN TRUE
    WHEN is_account_group_member('riesgo') THEN TRUE
    WHEN is_account_group_member('auditoria') THEN TRUE
    ELSE FALSE
  END;

In [0]:
%sql
SHOW CATALOGS LIKE 'fintech_finpay';

In [0]:
%sql
SHOW SCHEMAS IN fintech_finpay;

In [0]:
%sql
SHOW VOLUMES IN fintech_finpay.default;